# Using Claude as a RAG Evaluation Judge

Evaluating RAG pipelines is one of the hardest problems in production AI. This cookbook shows how to use Claude to measure the quality of a RAG system across four key dimensions:

| Metric | What it measures |
|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved context? |
| **Answer Relevancy** | Does the answer actually address the question asked? |
| **Context Precision** | Is the retrieved context useful for answering the question? |
| **Hallucination Detection** | Which specific claims, if any, are fabricated or unsupported? |

We'll build a complete evaluation harness using only the Anthropic Python SDK, including structured hallucination detection with citation-level analysis.

## Setup

Install the Anthropic SDK and set your API key.

In [ ]:
%pip install anthropic --quiet

In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

import anthropic
import pandas as pd

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-sonnet-4-6"

## Sample RAG Dataset

Each item contains a question, the context retrieved by the RAG system, the generated answer, and the ground truth answer. We include one item that contains a hallucination so we can verify the detector works.

In [ ]:
eval_dataset = [
    {
        "question": "What is retrieval-augmented generation?",
        "context": (
            "Retrieval-Augmented Generation (RAG) is an AI framework that combines a retrieval system "
            "with a generative language model. The retrieval component searches a knowledge base for "
            "relevant documents, which are then passed to the generative model as context to produce "
            "a grounded response."
        ),
        "answer": (
            "RAG is a technique that retrieves relevant documents from a knowledge base and uses them "
            "to help a language model generate more accurate, grounded responses."
        ),
        "ground_truth": (
            "RAG combines information retrieval with language generation to produce factually grounded responses."
        ),
    },
    {
        "question": "What are the main components of a transformer architecture?",
        "context": (
            "The transformer architecture, introduced in 'Attention is All You Need', consists of an encoder "
            "and decoder. Key components include multi-head self-attention mechanisms, position-wise "
            "feed-forward networks, positional encodings, and layer normalization."
        ),
        "answer": (
            "Transformers use self-attention, feed-forward layers, and positional encodings. "
            "They were originally developed at Google Brain in 2015."
        ),
        "ground_truth": (
            "Transformers consist of encoder/decoder blocks with multi-head attention, feed-forward networks, "
            "and positional encodings."
        ),
    },
    {
        "question": "How does vector search work?",
        "context": (
            "Vector databases store embeddings — numerical representations of data. When a query arrives, "
            "it is converted to an embedding and compared against stored embeddings using similarity metrics "
            "like cosine similarity or dot product. The most similar vectors are returned as results."
        ),
        "answer": (
            "Vector search converts queries to embeddings and finds the closest stored embeddings "
            "using cosine similarity."
        ),
        "ground_truth": (
            "Vector search converts text to numerical embeddings and retrieves the most similar "
            "embeddings using distance metrics."
        ),
    },
]

# Item 2 (index 1) contains a hallucination:
# "developed at Google Brain in 2015" is not in the context and is factually wrong
# (it was 2017, and by Google Brain + Google Research + University of Toronto)

print(f"Dataset loaded: {len(eval_dataset)} items")
print(f"Note: item #2 contains a deliberate hallucination for testing purposes.")

## Metric 1: Faithfulness

Measures whether the generated answer is grounded in the retrieved context — i.e., no claims beyond what the context supports.

Returns a score between 0.0 and 1.0.

In [ ]:
def evaluate_faithfulness(question: str, context: str, answer: str) -> float:
    """Score how faithful the answer is to the retrieved context.

    Returns a float in [0.0, 1.0] where 1.0 means every claim is
    directly supported by the context.
    """
    prompt = f"""You are an expert evaluator for RAG systems.

Given a question, retrieved context, and generated answer, evaluate whether the answer
is faithful to the context (i.e., it only makes claims that are supported by the context).

Question: {question}
Context: {context}
Answer: {answer}

Faithfulness score criteria:
1.0 = Every claim in the answer is directly supported by the context
0.5 = Most claims are supported, but some minor unsupported statements
0.0 = The answer contains significant claims not found in the context

Respond with ONLY a number between 0.0 and 1.0."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=10,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )

    try:
        return float(response.content[0].text.strip())
    except ValueError:
        return 0.0

## Metric 2: Answer Relevancy

Measures whether the generated answer actually addresses the question asked. An answer can be faithful to the context but still fail to answer the question.

Returns a score between 0.0 and 1.0.

In [ ]:
def evaluate_answer_relevancy(question: str, answer: str) -> float:
    """Score how well the answer addresses the question.

    Returns a float in [0.0, 1.0] where 1.0 means the answer
    directly and completely addresses the question.
    """
    prompt = f"""You are an expert evaluator for RAG systems.

Given a question and an answer, evaluate how well the answer addresses the question.

Question: {question}
Answer: {answer}

Answer relevancy criteria:
1.0 = The answer directly and completely addresses the question
0.5 = The answer partially addresses the question or goes off-topic
0.0 = The answer does not address the question at all

Respond with ONLY a number between 0.0 and 1.0."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=10,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )

    try:
        return float(response.content[0].text.strip())
    except ValueError:
        return 0.0

## Metric 3: Context Precision

Measures whether the retrieved context is actually useful for answering the question. A retrieval system might return semantically similar but ultimately unhelpful chunks.

Returns a score between 0.0 and 1.0.

In [ ]:
def evaluate_context_precision(question: str, context: str) -> float:
    """Score how relevant the retrieved context is to the question.

    Returns a float in [0.0, 1.0] where 1.0 means the context contains
    exactly what is needed to answer the question.
    """
    prompt = f"""You are an expert evaluator for RAG systems.

Given a question and retrieved context, evaluate how relevant and useful the context
is for answering the question.

Question: {question}
Context: {context}

Context precision criteria:
1.0 = The context contains exactly what is needed to answer the question
0.5 = The context is somewhat relevant but contains a lot of noise
0.0 = The context is irrelevant to the question

Respond with ONLY a number between 0.0 and 1.0."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=10,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )

    try:
        return float(response.content[0].text.strip())
    except ValueError:
        return 0.0

## Metric 4: Hallucination Detection

This metric goes beyond the faithfulness scalar score. It identifies **specific claims** in the answer that are not supported by the retrieved context and classifies each as either `supported` or `hallucinated`.

The key insight: faithfulness tells you *whether* hallucinations exist; hallucination detection tells you *exactly what* was hallucinated. This is critical for debugging and improving your RAG pipeline.

We use Claude's structured JSON output mode to get machine-readable claim-level analysis.

In [ ]:
def detect_hallucinations(question: str, context: str, answer: str) -> dict:
    """Identify specific hallucinated claims in the answer.

    Splits the answer into individual claims and classifies each as
    'supported' (grounded in context) or 'hallucinated' (not in context).

    Returns a dict with:
      - claims: list of {claim, verdict, explanation}
      - hallucination_rate: fraction of claims that are hallucinated
      - has_hallucinations: bool
    """
    prompt = f"""You are an expert fact-checker for RAG systems.

Your task is to identify which specific claims in the generated answer are supported by the
retrieved context and which are hallucinated (not found in or contradicted by the context).

Question: {question}

Retrieved Context:
{context}

Generated Answer:
{answer}

Instructions:
1. Break the answer into individual atomic claims (each claim = one verifiable fact or statement)
2. For each claim, check whether it is explicitly supported by the retrieved context
3. A claim is 'hallucinated' if it asserts something not present in or contradicted by the context
4. A claim is 'supported' if the context directly confirms it

Respond with a JSON object in this exact format:
{{
  "claims": [
    {{
      "claim": "the exact claim text from the answer",
      "verdict": "supported" | "hallucinated",
      "explanation": "brief reason: which part of the context supports or contradicts this"
    }}
  ]
}}

Output ONLY valid JSON. No markdown, no code fences, no extra text."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )

    try:
        data = json.loads(response.content[0].text.strip())
        claims = data.get("claims", [])
        hallucinated = [c for c in claims if c.get("verdict") == "hallucinated"]
        rate = len(hallucinated) / len(claims) if claims else 0.0
        return {
            "claims": claims,
            "hallucination_rate": round(rate, 3),
            "has_hallucinations": len(hallucinated) > 0,
        }
    except (json.JSONDecodeError, KeyError):
        return {
            "claims": [],
            "hallucination_rate": 0.0,
            "has_hallucinations": False,
        }

## Test: Hallucination Detection in Isolation

Before running the full evaluation, let's verify the detector catches the hallucination in item #2 ("developed at Google Brain in 2015" — not in context, and factually wrong).

In [ ]:
# Run hallucination detection on item #2 which contains a deliberate hallucination
item = eval_dataset[1]
result = detect_hallucinations(item["question"], item["context"], item["answer"])

print(f"Question: {item['question']}")
print(f"Answer:   {item['answer']}")
print(f"\nHallucination rate: {result['hallucination_rate']:.0%}")
print(f"Has hallucinations: {result['has_hallucinations']}")
print(f"\nClaim-level analysis:")
for c in result["claims"]:
    flag = "[HALL]" if c["verdict"] == "hallucinated" else "[ OK ]"
    print(f"  {flag}  {c['claim']}")
    print(f"         Reason: {c['explanation']}")

## Run Full Evaluation

Now evaluate all items across all four metrics.

In [ ]:
def evaluate_item(item: dict) -> dict:
    """Run all four metrics for a single dataset item concurrently.

    Uses a thread pool to fire all API calls in parallel, reducing wall-clock
    time from ~N*latency to ~1*latency per item.
    """
    with ThreadPoolExecutor(max_workers=4) as pool:
        f_future = pool.submit(evaluate_faithfulness, item["question"], item["context"], item["answer"])
        r_future = pool.submit(evaluate_answer_relevancy, item["question"], item["answer"])
        p_future = pool.submit(evaluate_context_precision, item["question"], item["context"])
        h_future = pool.submit(detect_hallucinations, item["question"], item["context"], item["answer"])

        faithfulness = f_future.result()
        relevancy = r_future.result()
        precision = p_future.result()
        hallucination = h_future.result()

    return {
        "question": item["question"],
        "faithfulness": faithfulness,
        "answer_relevancy": relevancy,
        "context_precision": precision,
        "hallucination_rate": hallucination["hallucination_rate"],
        "hallucinated_claims": [c for c in hallucination["claims"] if c["verdict"] == "hallucinated"],
        "average": round((faithfulness + relevancy + precision) / 3, 3),
    }


results = []

# Evaluate all items in parallel (one thread per item, metrics within each item also concurrent)
with ThreadPoolExecutor(max_workers=len(eval_dataset)) as pool:
    futures = {pool.submit(evaluate_item, item): i for i, item in enumerate(eval_dataset)}
    for future in as_completed(futures):
        i = futures[future]
        q_short = eval_dataset[i]["question"][:55] + "..." if len(eval_dataset[i]["question"]) > 55 else eval_dataset[i]["question"]
        print(f"[{i + 1}/{len(eval_dataset)}] Done: {q_short}")
        results.append(future.result())

# Sort back to original order
results.sort(key=lambda r: next(i for i, item in enumerate(eval_dataset) if item["question"] == r["question"]))
print("\nEvaluation complete.")

## Results Summary

In [ ]:
df = pd.DataFrame([
    {
        "Question": r["question"][:60] + "..." if len(r["question"]) > 60 else r["question"],
        "Faithfulness": r["faithfulness"],
        "Answer Relevancy": r["answer_relevancy"],
        "Context Precision": r["context_precision"],
        "Hallucination %": f"{r['hallucination_rate']:.0%}",
        "Average": r["average"],
    }
    for r in results
])

# Append an averages row
avg_row = pd.DataFrame([{
    "Question": "AVERAGE",
    "Faithfulness": round(df["Faithfulness"].mean(), 3),
    "Answer Relevancy": round(df["Answer Relevancy"].mean(), 3),
    "Context Precision": round(df["Context Precision"].mean(), 3),
    "Hallucination %": "",
    "Average": round(df["Average"].mean(), 3),
}])

display(pd.concat([df, avg_row], ignore_index=True))

In [ ]:
# Hallucination drill-down: show flagged claims for any item with hallucinations
flagged = [r for r in results if r["hallucination_rate"] > 0]

if flagged:
    print("\n=== Hallucination Drill-Down ===")
    for r in flagged:
        print(f"\nQuestion: {r['question']}")
        print(f"Hallucinated claims ({len(r['hallucinated_claims'])}):\n")
        for c in r["hallucinated_claims"]:
            print(f"  Claim:       {c['claim']}")
            print(f"  Explanation: {c['explanation']}")
            print()
else:
    print("\nNo hallucinations detected across the dataset.")

In [ ]:
## Export results to JSON
output_path = "eval_results.json"

# Serialize results (exclude non-serializable claim dicts from the summary export)
export = [
    {
        "question": r["question"],
        "faithfulness": r["faithfulness"],
        "answer_relevancy": r["answer_relevancy"],
        "context_precision": r["context_precision"],
        "hallucination_rate": r["hallucination_rate"],
        "hallucinated_claims": r["hallucinated_claims"],
        "average": r["average"],
    }
    for r in results
]

with open(output_path, "w") as f:
    json.dump(export, f, indent=2)

print(f"Results saved to {output_path}")

## Understanding the Metrics

### Faithfulness vs. Hallucination Detection

These two metrics are complementary, not redundant:

- **Faithfulness** gives you a scalar quality score (0.0–1.0). It's fast and useful for ranking and regression testing.
- **Hallucination Detection** gives you a structured breakdown of *which* claims are problematic. It's slower but essential for debugging.

In practice, use faithfulness for automated CI pipelines and hallucination detection for human review queues.

### Why Use LLM-as-Judge?

Traditional NLP metrics (BLEU, ROUGE, BERTScore) measure surface-level similarity. They can't tell you whether a claim is grounded in a source document. Claude can, because it understands meaning.

The tradeoff: LLM-as-judge is slower and more expensive than string-matching metrics, but far more accurate on semantic tasks like factual grounding.

## Best Practices

1. **Run on at least 20–50 samples.** Small datasets produce unreliable averages. Run on a representative sample of your production traffic.

2. **Use `temperature=0` for all evaluation calls.** You want deterministic scores. Variance in evaluation scores makes regression tracking meaningless.

3. **Use a different model for evaluation than generation.** Claude evaluating its own Claude-generated responses can introduce systematic bias. If your pipeline uses Claude for generation, consider using a different temperature, system prompt, or model tier for evaluation.

4. **Run faithfulness as a fast filter, then hallucination detection on failures.** Faithfulness is cheap (one API call, 10 tokens). Hallucination detection is more expensive. Use faithfulness to flag low-scoring items, then run claim-level detection only on those.

5. **Track all four scores over time.** The real value is detecting regressions when you update your retrieval system, chunk strategy, or generation prompt. A single snapshot is almost useless; the trend is the signal.

6. **Combine with human review.** Use these scores to triage samples for human annotation, not as a replacement for it. LLM judges can have systematic blind spots — especially for domain-specific claims they're not well-calibrated on.

7. **Log the raw claim-level output.** Even when a run shows zero hallucinations, save the full `detect_hallucinations()` output. It's useful for audits and for understanding what the system *almost* hallucinated.

## Next Steps

- Add a **Context Recall** metric: given the ground truth answer, how much of the required information was present in the retrieved context? This helps diagnose retrieval failures vs. generation failures.
- Add **citation grounding**: modify the generation step to require citations, then verify that each citation maps to the actual retrieved chunk.
- Integrate with your CI pipeline: run this evaluation on a held-out test set after every change to your retrieval configuration.